# 03rl · FunnyBirds visibility-aware labels — causal training test for CBM and MCBM

The standard training label `c_j` is fixed by species, even when part `j` is hidden. The relabeled training target is

`c_j^RL = c_j × visible_part(j)`.

CBM-RL and MCBM-RL use the same images, architecture, optimizer, seeds, and evaluation swaps as their standard counterparts. Only the training concept labels change.

If relabeling increases `P(z_donor,cf > z_source,cf)` on the same counterfactual swaps, the original label/visibility conflict caused part of the grounding failure. If it does not, that particular label correction was insufficient; it does not prove grounding failure is “intrinsic to all bottlenecks.”


In [ ]:
import os,re,glob
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
CURATED=Path(os.environ["CURATED_DATA"]);CWD=Path.cwd();REPO=CWD if (CWD/"analysis").is_dir() else CWD.parent
import sys;sys.path.insert(0,str(REPO/"analysis"))
try:
 from plotting import set_paper_style,PALETTE;set_paper_style();CBM_C,MCBM_C=PALETTE["CBM"],PALETTE["MCBM"]
except Exception:CBM_C,MCBM_C="#0072B2","#D55E00"
RL_C="#009E73";ORDER=["tail","wing","beak","foot","eye"];EPS=1e-3
def load_cbm(prefix):
 rows=[]
 for fp in sorted(glob.glob(str(CURATED/"swap"/f"{prefix}-s*.csv"))):
  m=re.search(r"-s(\d+)\.csv$",Path(fp).name)
  if m:d=pd.read_csv(fp);d["seed"]=int(m.group(1));rows.append(d)
 return pd.concat(rows,ignore_index=True) if rows else None
def load_mcbm(prefix):
 rows=[]
 for fp in sorted(glob.glob(str(CURATED/"swap"/f"{prefix}-g*-s*.csv"))):
  m=re.search(r"-g([0-9p]+)-s(\d+)",Path(fp).name)
  if m:d=pd.read_csv(fp);d["gamma"]=float(m.group(1).replace("p","."));d["seed"]=int(m.group(2));rows.append(d)
 return pd.concat(rows,ignore_index=True) if rows else None


## 1 · Standard CBM versus CBM-RL

For each part, calculate

`swap_following = P(margin>0)`, where `margin=z_donor,cf−z_source,cf`.

The standard and relabeled models are evaluated on identical swap logic. A repeated increase across seeds estimates the causal effect of changing training labels, provided the checkpoints differ only in those labels. Show every seed; rows within a seed reuse images and are not independent.


In [ ]:
CBM=load_cbm("funnybirds-cbm");CBMRL=load_cbm("funnybirds-cbm-rlv2")
if CBMRL is None:print("[pending] SEEDS='1 2 3' bash train/cbm_funnybirds_rl.sh; then run renderer_swap with CONFIG_PREFIX=funnybirds-cbm-rlv2")
else:
 rows=[]
 for name,d in [("standard",CBM),("relabeled",CBMRL)]:
  if d is None:continue
  q=d.groupby(["seed","part"]).ordering_correct.mean().reset_index();q["labels"]=name;rows.append(q)
 D=pd.concat(rows);display(D.round(3));G=D.groupby(["labels","part"]).ordering_correct.agg(["mean","std","count"])
 fig,ax=plt.subplots(figsize=(8,3.6));x=np.arange(len(ORDER));w=.38
 for off,(name,col) in zip([-.5,.5],[("standard",CBM_C),("relabeled",RL_C)]):
  q=G.loc[name].reindex(ORDER);ax.bar(x+off*w,q["mean"],w,yerr=q["std"].fillna(0),capsize=3,label=name,color=col)
 ax.axhline(.5,ls=":",color="gray");ax.set_xticks(x);ax.set_xticklabels(ORDER);ax.set_ylim(0,1)
 ax.set_ylabel("P(z_donor,cf > z_source,cf)");ax.set_title("CBM: effect of visibility-aware training labels");ax.legend();plt.show()


**Decision rule.** A higher relabeled bar means the training-label change improved grounding for that part. A remaining gap to 1 means the correction did not solve everything. No difference means this relabeling did not change swap-following; it does not establish that no data or architectural intervention could help.


## 2 · Standard MCBM versus MCBM-RL across `γ`

Repeat the same comparison at each minimality weight:

`Δ_RL(γ)=P(margin>0 | relabeled,γ)−P(margin>0 | standard,γ)`.

This asks whether visibility-aware labels help after `z_j` is also pulled toward `±3`. The label effect and the minimality effect are separate axes; `γ=0` MCBM is not the standard CBM.


In [ ]:
STD=load_mcbm("funnybirds-mcbm");RL=load_mcbm("funnybirds-mcbm-rlv2")
if RL is None:print("[pending] relabeled MCBM swap CSVs")
else:
 rows=[]
 for name,d in [("standard",STD),("relabeled",RL)]:
  if d is None:continue
  q=d[d.part=="tail"].groupby(["gamma","seed"]).ordering_correct.mean().reset_index();q["labels"]=name;rows.append(q)
 D=pd.concat(rows);display(D.round(3));G=D.groupby(["labels","gamma"]).ordering_correct.agg(["mean","std","count"])
 fig,ax=plt.subplots(figsize=(7,3.7))
 for name,col,mark in [("standard",MCBM_C,"o"),("relabeled",RL_C,"s")]:
  q=G.loc[name].reset_index();x=q.gamma.replace(0,.03);ax.errorbar(x,q["mean"],yerr=q["std"].fillna(0),marker=mark,label=name,color=col)
 ax.set_xscale("log");ax.axhline(.5,ls=":",color="gray");ax.set_ylim(0,1);ax.set_xlabel("γ (0 shown at 0.03)")
 ax.set_ylabel("tail P(z_donor,cf > z_source,cf)");ax.set_title("MCBM: effect of visibility-aware labels");ax.legend();plt.show()


Read each `γ` together with its seed count. If improvement occurs for both CBM and MCBM, the label/visibility conflict is not specific to minimality. If it occurs only for one model, the training objective interacts with the corrected labels.


## 3 · Independent deletion check

For visibly present parts, compute

`retained_frac = mean(p_removed)/mean(p_intact)`.

Lower `retained_frac` after relabeling means the canonical concept depends more strongly on its pixels. Deletion is a different intervention from replacement, so agreement strengthens the conclusion. It remains important to exclude no-op deletions using `changed_frac>0`.


In [ ]:
def retention(pattern,part="tail"):
 out=[]
 for fp in sorted(glob.glob(str(CURATED/"grounding"/pattern))):
  d=pd.read_parquet(fp);d=d[d.part==part]
  if "changed_frac" in d:d=d[d.changed_frac>EPS]
  m=re.search(r"-s(\d+)",Path(fp).stem);g=re.search(r"-g([0-9p]+)",Path(fp).stem)
  if len(d) and d.p_intact.mean()>1e-6:out.append(dict(seed=int(m.group(1)),gamma=float(g.group(1).replace("p",".")) if g else np.nan,retained_frac=d.p_removed.mean()/d.p_intact.mean()))
 return pd.DataFrame(out)
rows=[]
for model,labels,pat in [("CBM","standard","funnybirds-cbm-s*.parquet"),("CBM","relabeled","funnybirds-cbm-rlv2-s*.parquet"),("MCBM","standard","funnybirds-mcbm-g*-s*.parquet"),("MCBM","relabeled","funnybirds-mcbm-rlv2-g*-s*.parquet")]:
 q=retention(pat)
 if len(q):q["model"]=model;q["labels"]=labels;rows.append(q)
if rows:
 R=pd.concat(rows,ignore_index=True);display(R.round(3))
else:print("[pending] grounding parquets for relabeled checkpoints")


**Interpretation.** Swap improvement shows that the visible replacement more often defeats the old answer. Lower deletion retention shows that removing the original part removes more of its support. Together they identify the fraction of grounding failure changed by the training-label intervention.


## Conclusion

This notebook estimates a causal training-label effect because standard and relabeled runs differ only in `c_j` versus `c_j^RL`, while images, architecture, optimizer, seeds, and tests are held fixed.

It does not claim that relabeling is a complete solution. If grounding improves partially, the label/visibility conflict caused that portion and residual failure needs another explanation. If it does not improve, the specific absent-label rule may be insufficient—for example, an occluded concept might be better represented as “unknown/not visible” rather than semantically absent.
